# Master Thesis — Execution Pipeline

Runs the modelling-to-results pipeline in dependency order, skipping any stage whose outputs are already fresh.

**Stages**
1. **Modelling** — cleaned data → fitted models + modelled series (solar, temperature, copula, capture rate)
2. **Simulation** — model params → Monte Carlo innovations + physical variables
3. **Results** — simulated data → revenue index outputs

**Excluded**: Transformation notebooks and Energy price model.

**Staleness rule**: a stage re-runs if any output file is missing, or if any input file is newer than the oldest output.  
If a stage runs, all downstream stages are also marked stale (cascade).  
Set `FORCE_RERUN = True` to ignore all staleness checks.

In [7]:
import subprocess, sys, time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Configuration
Edit the flags below before running the pipeline.

In [8]:
# ── Run behaviour ──────────────────────────────────────────────────────────────
FORCE_RERUN      = False   # True: run all stages regardless of staleness
DRY_RUN          = False   # True: print plan only, do not execute notebooks
NOTEBOOK_TIMEOUT = 7200    # max seconds per notebook (2 h); increase for large sims

# Run a subset of stages (0-based indices), or None for all.
# Example: STAGE_FILTER = [2, 3]  → run only Simulation and Results
STAGE_FILTER = None

In [9]:
# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT = Path('..').resolve()   # project root (one level above Code/)
CODE = ROOT / 'Code'
DATA = ROOT / 'Data'

print(f'Project root : {ROOT}')
print(f'Python       : {sys.executable}')
print()

# ── Stage definitions ──────────────────────────────────────────────────────────
# notebooks : paths relative to Code/; executed in the listed order
# inputs    : files whose mtime is compared against output mtimes
# outputs   : files that must exist and be up-to-date; missing → stale

STAGES = [
    {
        'name': '1 - Modelling',
        'notebooks': [
            'Modelling/Solar Irradiation model.ipynb',
            'Modelling/Temperature model.ipynb',
            'Modelling/Solar share historical.ipynb',
            'Modelling/Covariance analysis.ipynb',
            'Modelling/Capture Price Model.ipynb',
        ],
        'inputs': [
            DATA / 'Cleaned' / 'df_bologna_cleaned.parquet',
        ],
        'outputs': [
            DATA / 'Modelled'  / 'df_solar_modelled.parquet',
            DATA / 'Modelled'  / 'df_temperature_modeled.parquet',
            DATA / 'Modelled'  / 'df_cr_modelled.parquet',
            DATA / 'Cleaned'   / 'df_solar_share.parquet',
            CODE / 'Models'    / 'GHI - KT'    / 'ghi_arma_result.pkl',
            CODE / 'Models'    / 'Temperature'  / 'temp_model_params.pkl',
            CODE / 'Models'    / 'Copula'       / 'copula_params.pkl',
            CODE / 'Models'    / 'Capture rate' / 'cr_ar_params.pkl',
        ],
    },
    {
        'name': '2 - Scenario Paths',
        'notebooks': [
            'Modelling/temperature paths.ipynb',
            'Modelling/Solar share paths.ipynb',
        ],
        'inputs': [
            CODE / 'Models' / 'Temperature' / 'temp_model_params.pkl',
            DATA / 'Cleaned' / 'df_solar_share.parquet',
        ],
        'outputs': [
            DATA / 'Cleaned' / 'df_temperature_scenarios.parquet',
            DATA / 'Cleaned' / 'df_penetration_scenarios.parquet',
        ],
    },
    {
        'name': '3 - Simulation',
        'notebooks': [
            'Simulation/Raw inovations simulation.ipynb',
            'Simulation/Variables reconstruction.ipynb',
            'Simulation/Variables reconstruction - multi-scenario.ipynb',
        ],
        'inputs': [
            CODE / 'Models' / 'GHI - KT'    / 'ghi_arma_result.pkl',
            CODE / 'Models' / 'Temperature'  / 'temp_model_params.pkl',
            CODE / 'Models' / 'Copula'       / 'copula_params.pkl',
            CODE / 'Models' / 'Capture rate' / 'cr_ar_params.pkl',
            DATA / 'Modelled' / 'df_solar_modelled.parquet',
            DATA / 'Modelled' / 'df_temperature_modeled.parquet',
            DATA / 'Modelled' / 'df_cr_modelled.parquet',
            DATA / 'Cleaned'  / 'df_penetration_scenarios.parquet',
        ],
        'outputs': [
            DATA / 'Simulated' / 'mc_innovations.parquet',
            DATA / 'Simulated' / 'mc_physical_variables.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_current_trend.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_pniec.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_entso_e.parquet',
        ],
    },
    {
        'name': '4 - Results',
        'notebooks': [
            'Modelling/Revenue index.ipynb',
            'Simulation/Revenue index - multi-scenario.ipynb',
        ],
        'inputs': [
            DATA / 'Simulated' / 'mc_physical_variables.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_current_trend.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_pniec.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_entso_e.parquet',
            DATA / 'Cleaned'   / 'df_temperature_scenarios.parquet',
            DATA / 'Modelled'  / 'df_cr_modelled.parquet',
            DATA / 'Modelled'  / 'df_temperature_modeled.parquet',
            DATA / 'Cleaned'   / 'df_bologna_cleaned.parquet',
        ],
        'outputs': [
            DATA / 'Results' / 'ri_annual_results.parquet',
            DATA / 'Results' / 'ri_annual_results_hist.parquet',
            DATA / 'Results' / 'ri_annual_results_multiscenario.parquet',
            DATA / 'Results' / 'risk_metrics_summary.parquet',
        ],
    },
]

print(f'Stages defined: {len(STAGES)}')
total_nbs = sum(len(s['notebooks']) for s in STAGES)
print(f'Notebooks total: {total_nbs}')

Project root : C:\Users\LucasMonero\Documents\data projects\Master Thesis\Project
Python       : c:\Users\LucasMonero\Documents\data projects\Master Thesis\Project\.venv\Scripts\python.exe

Stages defined: 4
Notebooks total: 12


## Helper functions

In [10]:
def _mtime(path):
    p = Path(path)
    return p.stat().st_mtime if p.exists() else 0.0


def is_stale(stage, upstream_reran=False):
    """Return (stale: bool, reason: str)."""
    if upstream_reran:
        return True, 'upstream stage reran'

    outputs = [Path(p) for p in stage.get('outputs', [])]
    inputs  = [Path(p) for p in stage.get('inputs',  [])]

    missing = [p.name for p in outputs if not p.exists()]
    if missing:
        return True, 'missing output(s): ' + ', '.join(missing)

    if outputs and inputs:
        existing_in = [p for p in inputs if p.exists()]
        if existing_in:
            oldest_out = min(_mtime(p) for p in outputs)
            newest_in  = max(_mtime(p) for p in existing_in)
            if newest_in > oldest_out:
                newer = [p.name for p in existing_in if _mtime(p) > oldest_out]
                return True, 'input(s) updated since last run: ' + ', '.join(newer)

    return False, 'up to date'


def run_notebook(nb_rel, timeout=NOTEBOOK_TIMEOUT):
    """Execute notebook in-place via nbconvert.
    nb_rel is a path relative to Code/.
    Returns (success: bool, elapsed_s: float, error_msg: str).
    """
    nb_path = CODE / nb_rel
    if not nb_path.exists():
        return False, 0.0, 'notebook not found: ' + str(nb_path)

    cmd = [
        sys.executable, '-m', 'jupyter', 'nbconvert',
        '--to', 'notebook',
        '--execute',
        '--inplace',
        f'--ExecutePreprocessor.timeout={timeout}',
        '--ExecutePreprocessor.kernel_name=python3',
        str(nb_path),
    ]

    t0 = time.time()
    proc = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd=str(nb_path.parent),   # kernel CWD = notebook directory
    )
    elapsed = time.time() - t0

    if proc.returncode != 0:
        raw = ((proc.stderr or '') + (proc.stdout or '')).strip()
        return False, elapsed, raw[-2000:]

    return True, elapsed, ''


def fmt_time(s):
    if s >= 60:
        return f'{s/60:.1f} min'
    return f'{s:.0f}s'


print('Helper functions ready.')

Helper functions ready.


## Run pipeline

In [ ]:
SEP = '=' * 68

print(SEP)
print(f'  PIPELINE  {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  FORCE_RERUN={FORCE_RERUN}  DRY_RUN={DRY_RUN}  '
      f'STAGE_FILTER={STAGE_FILTER if STAGE_FILTER is not None else "all"}')
print(SEP)

log            = []
upstream_reran = False
pipeline_ok    = True

for i, stage in enumerate(STAGES):

    tag = f'[{i+1}/{len(STAGES)}]'

    # ── stage filter ──────────────────────────────────────────────────────────
    if STAGE_FILTER is not None and i not in STAGE_FILTER:
        print(f'\n{tag} {stage["name"]}  -->  FILTERED OUT')
        log.append({'stage': stage['name'], 'status': 'filtered', 'notebooks': []})
        upstream_reran = False
        continue

    # ── staleness check ───────────────────────────────────────────────────────
    stale, reason = is_stale(stage, upstream_reran=upstream_reran)

    if not FORCE_RERUN and not stale:
        print(f'\n{tag} {stage["name"]}  -->  UP TO DATE  ({reason})')
        log.append({'stage': stage['name'], 'status': 'skipped',
                    'reason': reason, 'notebooks': []})
        upstream_reran = False
        continue

    run_reason = reason if not FORCE_RERUN else 'FORCE_RERUN=True'
    print(f'\n{tag} {stage["name"]}  -->  RUNNING  ({run_reason})')

    stage_entry = {
        'stage': stage['name'], 'status': 'ran',
        'reason': run_reason, 'notebooks': [], 'success': True,
    }
    stage_ok = True

    for nb_rel in stage['notebooks']:
        nb_name = Path(nb_rel).name

        if DRY_RUN:
            print(f'  [DRY]  {nb_name}')
            stage_entry['notebooks'].append({'notebook': nb_name, 'status': 'dry_run'})
            continue

        print(f'  Running  {nb_name} ...', end='', flush=True)
        ok, elapsed, err = run_notebook(nb_rel)
        status = 'OK' if ok else 'FAILED'
        print(f'  [{status}]  {fmt_time(elapsed)}')

        nb_entry = {'notebook': nb_name, 'status': status,
                    'elapsed_s': round(elapsed, 1)}
        if not ok:
            nb_entry['error'] = err[-600:]
            print(f'  -- error tail --')
            print(err[-500:])
            print(f'  ----------------')
            stage_ok = False

        stage_entry['notebooks'].append(nb_entry)

        if not ok:
            print(f'  Stopping stage after failure in {nb_name}.')
            break

    stage_entry['success'] = stage_ok
    log.append(stage_entry)
    upstream_reran = stage_ok and not DRY_RUN

    if not stage_ok:
        print(f'\nPipeline halted at stage {stage["name"]}.')
        pipeline_ok = False
        break

print(f'\n{SEP}')
print(f'  {"DONE" if pipeline_ok else "FAILED"}  '
      f'{datetime.now().strftime("%H:%M:%S")}')
print(SEP)

  PIPELINE  2026-05-08 09:02:21
  FORCE_RERUN=False  DRY_RUN=False  STAGE_FILTER=all

[1/4] 1 - Modelling  -->  RUNNING  (input(s) updated since last run: df_bologna_cleaned.parquet)
  Running  Solar Irradiation model.ipynb ...  [OK]  12.7 min
  Running  Temperature model.ipynb ...  [OK]  1.9 min
  Running  Solar share historical.ipynb ...  [OK]  17s
  Running  Covariance analysis.ipynb ...  [OK]  1.0 min
  Running  Capture Price Model.ipynb ...  [OK]  3.4 min

[2/4] 2 - Scenario Paths  -->  RUNNING  (upstream stage reran)
  Running  temperature paths.ipynb ...  [OK]  28s
  Running  Solar share paths.ipynb ...  [OK]  19s

[3/4] 3 - Simulation  -->  RUNNING  (upstream stage reran)
  Running  Raw inovations simulation.ipynb ...  [OK]  1.6 min
  Running  Variables reconstruction.ipynb ...  [OK]  9.2 min
  Running  Variables reconstruction - multi-scenario.ipynb ...  [OK]  17.9 min

[4/4] 4 - Results  -->  RUNNING  (upstream stage reran)
  Running  Revenue index.ipynb ...  [OK]  8.7 min
  

## Summary

In [ ]:
print('PIPELINE SUMMARY')
print('-' * 55)

total_s = sum(
    nb.get('elapsed_s', 0)
    for entry in log
    for nb in entry.get('notebooks', [])
)

STATUS_TAG = {'ran': 'RAN', 'skipped': '---', 'filtered': '   '}

for entry in log:
    s   = entry['status']
    tag = STATUS_TAG.get(s, '???')
    ok  = '' if s != 'ran' else ('  OK' if entry.get('success') else '  FAILED')
    print(f'  [{tag}]  {entry["stage"]}{ok}')
    for nb in entry.get('notebooks', []):
        st = nb['status']
        ic = 'ok' if st == 'OK' else ('!!' if st == 'FAILED' else '..')
        t  = f'  ({fmt_time(nb["elapsed_s"])})' if 'elapsed_s' in nb else ''
        print(f'         [{ic}]  {nb["notebook"]}{t}')

print(f'\n  Total wall time : {fmt_time(total_s)}')

failed = [e['stage'] for e in log
          if e.get('status') == 'ran' and not e.get('success', True)]
if failed:
    print(f'  FAILED stages   : {failed}')
else:
    ran     = [e['stage'] for e in log if e.get('status') == 'ran']
    skipped = [e['stage'] for e in log if e.get('status') == 'skipped']
    if ran:
        print(f'  Ran             : {ran}')
    if skipped:
        print(f'  Skipped (fresh) : {skipped}')

PIPELINE SUMMARY
-------------------------------------------------------
  [RAN]  1 - Modelling  OK
         [ok]  Solar Irradiation model.ipynb  (7.0 min)
         [ok]  Temperature model.ipynb  (48s)
         [ok]  Solar share historical.ipynb  (10s)
         [ok]  Covariance analysis.ipynb  (32s)
         [ok]  Capture Price Model.ipynb  (1.3 min)
  [RAN]  2 - Scenario Paths  OK
         [ok]  temperature paths.ipynb  (10s)
         [ok]  Solar share paths.ipynb  (9s)
  [RAN]  3 - Simulation  FAILED
         [ok]  Raw inovations simulation.ipynb  (57s)
         [!!]  Variables reconstruction.ipynb  (38s)

  Total wall time : 11.6 min
  FAILED stages   : ['3 - Simulation']
